# Clase 209 — Prefect 3 + Dagster: el mismo pipeline 2 veces

Mismo pipeline BTC, implementado en Prefect y en Dagster. Comparación lado a lado.

In [ ]:
import os, tempfile, shutil, json
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'flows_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

## 1. Prefect 3 — flow + tasks

In [ ]:
from prefect import flow, task, get_run_logger
from datetime import datetime
import duckdb

DB = str(WORK / 'btc_prefect.duckdb')

@task(retries=3, retry_delay_seconds=5)
def extract() -> dict:
    # stub local: número fake en vez de API
    return {'ts': datetime.utcnow().isoformat(), 'usd': 67500.0}

@task
def load(payload: dict) -> str:
    con = duckdb.connect(DB)
    con.execute('CREATE TABLE IF NOT EXISTS prices (ts TIMESTAMP PRIMARY KEY, usd DOUBLE)')
    con.execute('INSERT OR IGNORE INTO prices VALUES (?, ?)', [payload['ts'], payload['usd']])
    con.close()
    return DB

@task
def transform(db_path: str) -> dict:
    con = duckdb.connect(db_path)
    avg = con.execute('SELECT AVG(usd), COUNT(*) FROM prices').fetchone()
    con.close()
    return {'avg_usd': float(avg[0]) if avg[0] else None, 'n': int(avg[1])}

@flow(name='btc-pipeline')
def btc_pipeline():
    log = get_run_logger()
    metrics = transform(load(extract()))
    log.info(f'metrics: {metrics}')
    return metrics

result = btc_pipeline()
print('resultado:', result)

## 2. Prefect deployment con schedule

In [ ]:
deploy_snippet = '''\
# deploy.py — corré con: python deploy.py
from prefect import serve
from btc_module import btc_pipeline   # importa tu flow

if __name__ == "__main__":
    deployment = btc_pipeline.to_deployment(
        name="btc-hourly",
        cron="0 * * * *",          # cada hora
        tags=["crypto", "prod"],
    )
    serve(deployment)   # mantiene proceso vivo y ejecuta según schedule
'''
print(deploy_snippet)
print('# UI: prefect server start   →   http://localhost:4200')

## 3. Dagster — software-defined assets

In [ ]:
dagster_src = '''\
# dagster_btc.py — corré con: dagster dev -f dagster_btc.py
from dagster import asset, Definitions, AssetExecutionContext, MetadataValue
from datetime import datetime
import duckdb

DB = "/tmp/btc_dagster.duckdb"

@asset
def btc_price() -> dict:
    """Precio actual de BTC scrapeado de la API."""
    return {"ts": datetime.utcnow().isoformat(), "usd": 67500.0}

@asset(deps=[btc_price])
def btc_table(context: AssetExecutionContext, btc_price: dict) -> str:
    """Tabla DuckDB con historial de precios."""
    con = duckdb.connect(DB)
    con.execute("CREATE TABLE IF NOT EXISTS prices (ts TIMESTAMP PRIMARY KEY, usd DOUBLE)")
    con.execute("INSERT OR IGNORE INTO prices VALUES (?, ?)", [btc_price["ts"], btc_price["usd"]])
    n = con.execute("SELECT COUNT(*) FROM prices").fetchone()[0]
    context.add_output_metadata({"rows": MetadataValue.int(n)})
    return DB

@asset(deps=[btc_table])
def daily_avg(context: AssetExecutionContext, btc_table: str) -> float:
    """Promedio diario de precio."""
    con = duckdb.connect(btc_table)
    avg = con.execute("SELECT AVG(usd) FROM prices").fetchone()[0]
    context.add_output_metadata({"avg_usd": MetadataValue.float(float(avg))})
    return float(avg)

defs = Definitions(assets=[btc_price, btc_table, daily_avg])
'''
print(dagster_src)

## 4. Comparativa: mismo pipeline, 3 estilos

In [ ]:
import pandas as pd
comp = pd.DataFrame([
    {'aspect': 'Líneas de código', 'Airflow': '~35 (DAG)', 'Prefect': '~25 (flow)', 'Dagster': '~30 (assets)'},
    {'aspect': 'Modelo mental', 'Airflow': 'task graph', 'Prefect': 'task graph', 'Dagster': 'asset graph'},
    {'aspect': 'Setup local', 'Airflow': 'docker-compose 4 servicios', 'Prefect': 'pip install + 1 cmd', 'Dagster': 'pip install + dagster dev'},
    {'aspect': 'UI / lineage', 'Airflow': 'task-level', 'Prefect': 'task-level + clean', 'Dagster': 'asset-level + freshness'},
    {'aspect': 'Hybrid execution', 'Airflow': 'manual', 'Prefect': 'nativo (workers)', 'Dagster': 'nativo (code locations)'},
    {'aspect': 'Ideal cuando', 'Airflow': 'industria, max madurez', 'Prefect': 'Python-first, equipo chico', 'Dagster': 'data products + dbt'},
])
print(comp.to_string(index=False))

## Ejercicio guiado

1. Levantá `prefect server start` y `python deploy.py`. Confirmá ejecuciones cada minuto (cambiá cron a `*/1 * * * *` para test).
2. Levantá `dagster dev -f dagster_btc.py`. UI en `localhost:3000`. Click "Materialize" sobre `btc_price` → `daily_avg` queda "stale".
3. Materializá `daily_avg`. Confirmá que Dagster re-corre la cadena entera (porque depende de stale).
4. Agregá un check `@asset_check` en Dagster que verifique `avg_usd > 0`. UI muestra el check verde/rojo al lado del asset.
5. Bonus: integrá `dagster-dbt` y agregá un modelo dbt SQL como asset downstream.

## Conclusiones

- Prefect 3 es Airflow refactorizado con Python idiomático moderno; bueno para empezar.
- Dagster cambia el modelo mental: pensás en **datos producidos**, no en **tareas ejecutadas**.
- Para equipos con muchos data products + dbt: Dagster gana.
- Para equipos chicos sin DevOps dedicado: Prefect Cloud free tier es la opción más fácil.

## ✅ Soluciones de los ejercicios

Prefect y Dagster no están instalados en el laboratorio. Estas soluciones **reproducen su
modelo mental en proceso**: un `@flow` de Prefect es una función Python normal que llama a
sub-tareas (por eso corre con `python archivo.py`, sin scheduler); un `@asset` de Dagster
es un nodo de un grafo de dependencias que se materializa en orden topológico. Simulamos
ambos con funciones puras y un mini-resolver de dependencias.

### Ejercicio 1 — Prefect flow (`notify(transform(load(extract())))`)

En Prefect un `@flow` es simplemente una función; las `@task` son las que llama. La gracia
es que **corre directo** sin infraestructura. Aquí lo emulamos 1:1.

In [ ]:
# --- Equivalente Prefect (referencia) --------------------------------------
# from prefect import flow, task
# @task def extract(): ...
# @flow def btc_pipeline(): return notify(transform(load(extract())))
# ---------------------------------------------------------------------------

def extract():
    return {"price": 67500.0, "ts": "2026-06-01T00:00:00"}

def load(payload):
    payload = dict(payload)
    payload["loaded"] = True
    return payload

def transform(payload):
    payload = dict(payload)
    payload["price_k"] = round(payload["price"] / 1000, 1)   # a miles de USD
    return payload

def notify(payload):
    return f"BTC = {payload['price_k']}k USD (loaded={payload['loaded']})"

def btc_pipeline():
    return notify(transform(load(extract())))

msg = btc_pipeline()      # <- corre directo, como `python btc.py`
print(msg)
assert msg == "BTC = 67.5k USD (loaded=True)"
print("OK ejercicio 1 — flow Prefect ejecutado en proceso, sin scheduler")

### Ejercicio 2 — Deployment con cron `0 * * * *` (cada hora)

`flow.serve(cron="0 * * * *")` programa una ejecución al minuto 0 de cada hora. Modelamos
el cálculo de las próximas N ejecuciones que el scheduler dispararía.

In [ ]:
from datetime import datetime, timedelta

def next_hourly_runs(start, n):
    """Cron '0 * * * *' -> minuto 0 de cada hora. Devuelve las próximas n."""
    # redondear hacia arriba a la próxima hora en punto
    base = start.replace(minute=0, second=0, microsecond=0)
    if start > base:
        base += timedelta(hours=1)
    return [base + timedelta(hours=i) for i in range(n)]

runs = next_hourly_runs(datetime(2026, 6, 1, 9, 17), n=4)
for r in runs:
    print("run programado:", r.isoformat())

assert all(r.minute == 0 for r in runs), "todas las ejecuciones al minuto 0"
assert (runs[1] - runs[0]) == timedelta(hours=1), "separación de 1 hora"
assert runs[0] == datetime(2026, 6, 1, 10, 0), "9:17 -> primera corrida a las 10:00"
print("OK ejercicio 2 — cron horario materializa 1 run por hora")

### Ejercicio 3 — Dagster assets (`daily_avg` depende de `btc_price`)

Dagster es *data-aware*: declarás **assets** (tablas/datasets) y sus dependencias por el
nombre de los argumentos. Simulamos el grafo y lo materializamos en orden topológico.

In [ ]:
class AssetGraph:
    def __init__(self):
        self.fns, self.deps, self.store, self.stale = {}, {}, {}, set()

    def asset(self, *deps):
        def deco(fn):
            self.fns[fn.__name__] = fn
            self.deps[fn.__name__] = list(deps)
            self.stale.add(fn.__name__)
            return fn
        return deco

    def materialize(self, name):
        for d in self.deps[name]:
            if d not in self.store or d in self.stale:
                self.materialize(d)
        kwargs = {d: self.store[d] for d in self.deps[name]}
        self.store[name] = self.fns[name](**kwargs)
        self.stale.discard(name)
        return self.store[name]

g = AssetGraph()

@g.asset()
def btc_price():
    return [67500.0, 68000.0, 66000.0, 69000.0]

@g.asset("btc_price")
def daily_avg(btc_price):
    return sum(btc_price) / len(btc_price)

val = g.materialize("daily_avg")   # Dagster resuelve btc_price primero
print("daily_avg =", val)
assert abs(val - 67625.0) < 1e-9
assert "btc_price" in g.store, "la dependencia se materializó automáticamente"
print("OK ejercicio 3 — grafo de assets materializado en orden topológico")

### Ejercicio 4 — Materializar un solo asset deja el downstream *stale*

Si re-materializás `btc_price`, Dagster marca `daily_avg` como **stale** (desactualizado)
hasta que también se materialice. Modelamos esa propagación de staleness.

In [ ]:
# nuevo dato de entrada -> re-materializar solo btc_price
g.stale.add("daily_avg")          # al cambiar un upstream, el downstream queda stale
g.fns["btc_price"] = lambda: [70000.0, 71000.0]
g.materialize("btc_price")        # materializamos SOLO el upstream

print("daily_avg está stale?", "daily_avg" in g.stale)
assert "daily_avg" in g.stale, "cambió btc_price -> daily_avg quedó stale"

g.materialize("daily_avg")        # ahora sí lo actualizamos
print("daily_avg actualizado =", g.store["daily_avg"])
assert "daily_avg" not in g.stale
assert g.store["daily_avg"] == 70500.0
print("OK ejercicio 4 — staleness propagada y resuelta al re-materializar")

### Ejercicio 5 — Comparativa Airflow vs Prefect vs Dagster

Mismo pipeline, tres frameworks. Resumimos las diferencias de diseño en una tabla y la
validamos programáticamente.

In [ ]:
import pandas as pd

comparativa = pd.DataFrame([
    {"framework": "Airflow", "modelo": "DAG de tasks", "corre_sin_server": False,
     "data_aware": False, "loc_relativo": 3, "feedback_dev": "lento"},
    {"framework": "Prefect", "modelo": "flow = función Python", "corre_sin_server": True,
     "data_aware": False, "loc_relativo": 1, "feedback_dev": "rápido"},
    {"framework": "Dagster", "modelo": "grafo de assets", "corre_sin_server": True,
     "data_aware": True, "loc_relativo": 2, "feedback_dev": "rápido"},
])
print(comparativa.to_string(index=False))

# Airflow necesita scheduler+DB; Prefect/Dagster corren un flow con `python file.py`.
assert comparativa.set_index("framework").loc["Airflow", "corre_sin_server"] == False
assert comparativa.set_index("framework").loc["Dagster", "data_aware"] == True
assert comparativa["loc_relativo"].idxmin() is not None
print("OK ejercicio 5 — comparativa de los 3 orquestadores")